# VoC and Customer Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### a. Load the dataset

Google Play Review Dataset

In [2]:
# read google play review data -> save dataframe to rev_df

rev_df = pd.read_csv('../data/raw/Google Play Reviews/reviews.csv')

Let's get a sense of the structure of our data..

### b. Data Exploration

- Explore columns
- Fix datatypes
- Check counts, etc.

#### i. Column and Types

In [3]:
rev_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12495 entries, 0 to 12494
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              12495 non-null  object
 1   userName              12495 non-null  object
 2   userImage             12495 non-null  object
 3   content               12495 non-null  object
 4   score                 12495 non-null  int64 
 5   thumbsUpCount         12495 non-null  int64 
 6   reviewCreatedVersion  10333 non-null  object
 7   at                    12495 non-null  object
 8   replyContent          5818 non-null   object
 9   repliedAt             5818 non-null   object
 10  sortOrder             12495 non-null  object
 11  appId                 12495 non-null  object
dtypes: int64(2), object(10)
memory usage: 1.1+ MB


In [4]:
# 1. check the shape of the data
rev_df.shape

(12495, 12)

There are 12 total columns, and 12,495 rows of data.

In [5]:
# 2. check the datatypes
rev_df.head(2)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,sortOrder,appId
0,gp:AOqpTOEhZuqSqqWnaKRgv-9ABYdajFUB0WugPGh-SG-...,Eric Tie,https://play-lh.googleusercontent.com/a-/AOh14...,I cannot open the app anymore,1,0,5.4.0.6,2020-10-27 21:24:41,NaN,NaN,newest,com.anydo
1,gp:AOqpTOH0WP4IQKBZ2LrdNmFy_YmpPCVrV3diEU9KGm3...,john alpha,https://play-lh.googleusercontent.com/a-/AOh14...,I have been begging for a refund from this app...,1,0,NaN,2020-10-27 14:03:28,"Please note that from checking our records, yo...",2020-10-27 15:05:52,newest,com.anydo


Some columns are do not have their right datatypes. Some of these datatypes are not memory optimal, and currently consuming 1MB+ of memory storage. 

Let's fix the datatype of the columns

In [6]:
# explore 'score' column
print('These are the unique values in "score" column', rev_df['score'].unique())

print('The datatype of this column is ', rev_df['score'].dtype)

These are the unique values in "score" column [1 2 3 4 5]
The datatype of this column is  int64


In [7]:
# explore 'at' column
print('The datatype of this column is', rev_df['at'].dtype)

# convert data type
rev_df['at'] = pd.to_datetime(rev_df['at'], errors='coerce')
print('The datatype of this column is', rev_df['at'].dtype)

The datatype of this column is object
The datatype of this column is datetime64[ns]


This is a datetime dtype. Let's convert it to its right type

In [9]:
# explore 'repliedAt' column
print('These are the unique values in "score" column', rev_df['repliedAt'].dtype)

# change repliedAt to datetime
rev_df['repliedAt'] = pd.to_datetime(rev_df['repliedAt'], errors='coerce')
print('These are the unique values in "score" column', rev_df['repliedAt'].dtype)

These are the unique values in "score" column object
These are the unique values in "score" column datetime64[ns]


In [10]:
# explore 'sortOrder' and 'appId' columns
print('Datatype for sortOrder is', rev_df['sortOrder'].dtype)

print('Datatype for appId is', rev_df['appId'].dtype)

Datatype for sortOrder is object
Datatype for appId is object


In [11]:
print('Unique values for sortOrder:', rev_df['sortOrder'].unique())
print('\nUnique values for sortOrder:', rev_df['appId'].unique())


Unique values for sortOrder: ['newest']

Unique values for sortOrder: ['com.anydo' 'com.todoist' 'com.ticktick.task'
 'com.habitrpg.android.habitica' 'cc.forestapp' 'com.oristats.habitbull'
 'com.levor.liferpgtasks' 'com.habitnow' 'com.microsoft.todos'
 'prox.lab.calclock' 'com.gmail.jmartindev.timetune'
 'com.artfulagenda.app' 'com.tasks.android' 'com.appgenix.bizcal'
 'com.appxy.planner']


Change these column dtypes to category

In [12]:
rev_df['sortOrder'] = rev_df['sortOrder'].astype('category')

rev_df['appId'] = rev_df['appId'].astype('category')

In [13]:
print(rev_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12495 entries, 0 to 12494
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              12495 non-null  object        
 1   userName              12495 non-null  object        
 2   userImage             12495 non-null  object        
 3   content               12495 non-null  object        
 4   score                 12495 non-null  int64         
 5   thumbsUpCount         12495 non-null  int64         
 6   reviewCreatedVersion  10333 non-null  object        
 7   at                    12495 non-null  datetime64[ns]
 8   replyContent          5818 non-null   object        
 9   repliedAt             5818 non-null   datetime64[ns]
 10  sortOrder             12495 non-null  category      
 11  appId                 12495 non-null  category      
dtypes: category(2), datetime64[ns](2), int64(2), object(6)
memory usage: 1001.

#### b. Missing Values

Check for missing values in each column

In [15]:
missing_values = round(rev_df.isnull().sum() * 100 / len(rev_df),2)

print(missing_values)

reviewId                 0.00
userName                 0.00
userImage                0.00
content                  0.00
score                    0.00
thumbsUpCount            0.00
reviewCreatedVersion    17.30
at                       0.00
replyContent            53.44
repliedAt               53.44
sortOrder                0.00
appId                    0.00
dtype: float64


For majority of the dataset, there are no nulls. But there are 3 columns with null values:
- reviewCreatedVersion
- replyContent
- repliedAt

We can work with these for now.

#### iii. Check duplicates

In [16]:
# Count the number of duplicate IDs
duplicate_id_count = rev_df.duplicated(subset=['reviewId']).sum()
print(f"Number of duplicate IDs: {duplicate_id_count}")

# Show the rows with duplicate IDs 
duplicate_ids = rev_df[rev_df.duplicated(subset=['reviewId'], keep='first')]
print("Rows with duplicate IDs (excluding first occurrences):")
print(duplicate_ids)


Number of duplicate IDs: 0
Rows with duplicate IDs (excluding first occurrences):
Empty DataFrame
Columns: [reviewId, userName, userImage, content, score, thumbsUpCount, reviewCreatedVersion, at, replyContent, repliedAt, sortOrder, appId]
Index: []


#### iv. Date Range

In [17]:
# Find the earliest date
earliest_date = rev_df['at'].min()
print(f"The earliest date is: {earliest_date}")

# Find the latest date
latest_date = rev_df['at'].max()
print(f"The latest date is: {latest_date}")


The earliest date is: 2015-02-08 13:58:47
The latest date is: 2020-10-28 01:44:01


In [18]:
# Continue for other datetime columns

# Find the earliest date
earliest_date = rev_df['repliedAt'].min()
print(f"The earliest date is: {earliest_date}")

# Find the latest date
latest_date = rev_df['repliedAt'].max()
print(f"The latest date is: {latest_date}")


The earliest date is: 2013-01-14 13:17:53
The latest date is: 2020-10-28 01:36:33


There is an inconsistency here... The earliest date is in 2013, whereas the earliest in the day the reviews were created is 2015. The row will be dropped if need be.

#### v. Language Check (if texts exist)

Check if all texts are in expected language

In [20]:
from langdetect import detect, LangDetectException

In [21]:
def detect_language(text):
    try:
        if not isinstance(text, str):
            return 'not a string'
        return detect(text)
    except LangDetectException:
        return 'unknown'

In [22]:
rev_df['detected_lang'] = rev_df['content'].apply(detect_language)

rev_df['detected_lang'].head()

0    en
1    en
2    en
3    en
4    en
Name: detected_lang, dtype: object

In [27]:
print(rev_df['detected_lang'].unique())

['en' 'tr' 'de' 'af' 'so' 'id' 'unknown' 'da' 'it' 'ro' 'cy' 'et' 'sw'
 'nl' 'sv' 'pt' 'no' 'ru' 'pl' 'es' 'sq' 'fi' 'fa' 'fr' 'sl' 'ko' 'vi'
 'tl' 'cs' 'hu' 'ca' 'bn' 'ar' 'sk' 'hr' 'lt' 'ur' 'ne' 'th' 'hi' 'zh-cn'
 'uk' 'bg' 'he' 'lv']


In [28]:
rev_df['detected_lang'] = rev_df['detected_lang'].astype('category')

In [29]:
print(rev_df['replyContent'].apply(detect_language).unique())

['not a string' 'en' 'cy' 'unknown' 'de' 'pt' 'ru' 'fr' 'zh-cn' 'af' 'uk'
 'es' 'id' 'tr' 'so' 'pl' 'bg' 'et' 'tl' 'nl' 'it']
